## Lab: Reinforcement Learning with Human Feedback (RLHF)

### University of Virginia
### Reinforcement Learning
#### Last updated: November 4, 2026

---

#### Instructions:

Carefully read the notes below and run the provided code. Then complete the questions in the **Questions** section at the end of the notebook. Answer each question clearly and show all results.

**Guidelines on AI tools:**  

All code should be your own. If you would like to get help from an LLM, you may use it as a tutor or coding assistant to explain concepts, help you debug, suggest approaches, or provide small examples. You should understand, review, and test any code that you use, and you should be able to explain how your submitted code works. Do not use an LLM to generate the entire solution and submit it without substantial review and understanding.

#### TOTAL POINTS: 10

---

#### Background

RLHF is how models like ChatGPT and Claude are aligned with what people actually want, when "what people want" is too fuzzy to write down as a reward function. The InstructGPT recipe has three stages:

1. **Supervised fine-tuning (SFT):** fine-tune a pretrained model on human-written demonstrations. This gives the *reference policy* $\pi_{\text{ref}}$.
2. **Reward modeling:** show humans pairs of model outputs, ask which is better, and fit a reward model $r_\phi(y)$ to those comparisons.
3. **RL fine-tuning:** optimize the policy $\pi_\theta$ to earn high reward from $r_\phi$ **while staying close to $\pi_{\text{ref}}$**:

$$\max_\theta \; \mathbb{E}_{y \sim \pi_\theta}\big[r_\phi(y)\big] \;-\; \beta \, \mathrm{KL}\big(\pi_\theta \,\|\, \pi_{\text{ref}}\big)$$

In this lab we build stages 2 and 3 in a toy world small enough to see every number:

- A "response" is a 3-token string over the vocabulary `{A, B, C}` (27 possible responses).
- A simulated human prefers responses that match `ABC` in more positions.
- **The policy never sees the human's scoring rule.** It only gets feedback through the learned reward model, just like a real LLM.

We skip SFT and use a uniform reference policy (every response equally likely). Part 4 shows what can go wrong when the reward model is wrong, and why the KL term matters.

---

#### Import Packages

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

#### Configs

In [ ]:
VOCAB = ["A", "B", "C"]
TARGET = ["A", "B", "C"]   # the response the simulated human likes best
SEQ_LEN = len(TARGET)
V = len(VOCAB)
TOKEN_ID = {t: i for i, t in enumerate(VOCAB)}

N_PAIRS = 1000     # number of human comparisons collected
RM_EPOCHS = 5      # reward-model training epochs
RM_LR = 0.05       # reward-model learning rate

---

### Part 1: Simulated Human Preferences

The "human" scores each response by how many positions match `TARGET`, then picks the higher-scoring one. When the two responses score the same, the human has no preference, so we **discard the pair**. (Forcing a choice on ties would add label noise; real RLHF pipelines often drop or down-weight ties too.)

In [ ]:
def generate_random_sequence(length=SEQ_LEN):
    return rng.choice(VOCAB, size=length).tolist()


def true_score(seq):
    """The human's hidden scoring rule: number of positions matching TARGET."""
    return sum(a == b for a, b in zip(seq, TARGET))


def human_preference(seq1, seq2, score_fn=true_score):
    """Return 0 if seq1 is preferred, 1 if seq2 is preferred, None for a tie."""
    s1, s2 = score_fn(seq1), score_fn(seq2)
    if s1 == s2:
        return None
    return 0 if s1 > s2 else 1


def collect_preferences(n_pairs, score_fn=true_score):
    """Collect n_pairs non-tied comparisons of random response pairs."""
    preferences = []
    while len(preferences) < n_pairs:
        s1, s2 = generate_random_sequence(), generate_random_sequence()
        pref = human_preference(s1, s2, score_fn)
        if pref is not None:
            preferences.append((s1, s2, pref))
    return preferences


preferences = collect_preferences(N_PAIRS)
print("Sample preference:", preferences[0])

---

### Part 2: Train a Reward Model from Preferences

The reward model is a table `R[position, token]`, and a response's reward is the sum of its entries:

$$r_\phi(y) = \sum_{i} R[i, y_i]$$

We fit it with the **Bradley-Terry** model used in InstructGPT. The probability that the human prefers $y_1$ over $y_2$ is

$$P(y_1 \succ y_2) = \sigma\big(r_\phi(y_1) - r_\phi(y_2)\big)$$

and we maximize the log-likelihood of the observed choices. The gradient has a clean form: each table entry moves by `lr * (label - P(y1 ≻ y2))` times (one-hot of $y_1$ minus one-hot of $y_2$). Positions where both responses have the same token cancel out, because a comparison says nothing about them.

In [ ]:
def one_hot(seq):
    x = np.zeros((SEQ_LEN, V))
    for i, token in enumerate(seq):
        x[i, TOKEN_ID[token]] = 1.0
    return x


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def sequence_reward(R, seq):
    return float((R * one_hot(seq)).sum())


def train_reward_model(preferences, epochs=RM_EPOCHS, lr=RM_LR):
    """Fit a fresh reward table to preference data. Returns the table."""
    R = rng.normal(0, 0.1, size=(SEQ_LEN, V))
    for _ in range(epochs):
        for s1, s2, pref in preferences:
            diff = one_hot(s1) - one_hot(s2)
            p1 = sigmoid((R * diff).sum())      # model's P(s1 preferred)
            label = 1.0 if pref == 0 else 0.0   # 1 if human preferred s1
            R += lr * (label - p1) * diff       # gradient ascent on log-likelihood
    return R


def reward_model_accuracy(R, preferences):
    """Fraction of comparisons where the reward model agrees with the human."""
    correct = [
        (sequence_reward(R, s1) > sequence_reward(R, s2)) == (pref == 0)
        for s1, s2, pref in preferences
    ]
    return float(np.mean(correct))


def show_table(R, title="Reward table"):
    print(title)
    print("        " + "  ".join(f"{t:>6}" for t in VOCAB))
    for i in range(SEQ_LEN):
        print(f"pos {i}:  " + "  ".join(f"{R[i, j]:6.2f}" for j in range(V)))


R = train_reward_model(preferences)
show_table(R)

held_out = collect_preferences(500)
print(f"\nHeld-out accuracy: {reward_model_accuracy(R, held_out):.3f}")

---

### Part 3: RL Fine-Tuning with a KL Penalty

Now the actual RL. The policy $\pi_\theta$ chooses each token independently: position $i$ has its own softmax over `{A, B, C}` with logits `theta[i]`. The reference policy is uniform.

Each training step:

1. Sample a batch of responses from $\pi_\theta$.
2. Score each one with the **reward model** (not the human).
3. Subtract the KL penalty for that sample: $\;\tilde r(y) = r_\phi(y) - \beta\,\big[\log \pi_\theta(y) - \log \pi_{\text{ref}}(y)\big]$
4. Take a REINFORCE step with the batch mean as a baseline: $\;\nabla_\theta \approx \frac{1}{N}\sum_n (\tilde r_n - \bar{\tilde r})\,\nabla_\theta \log \pi_\theta(y_n)$

For a softmax, $\nabla_{\theta_i} \log \pi_\theta(y) = \text{onehot}(y_i) - \pi_\theta(\cdot \mid i)$.

This is the same objective InstructGPT optimizes with PPO. We use plain REINFORCE because the problem is tiny.

In [ ]:
def softmax(logits):
    z = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


REF_PROBS = np.full((SEQ_LEN, V), 1.0 / V)   # uniform reference policy


def kl_to_ref(probs):
    """Exact KL(pi || ref), summed over positions."""
    return float((probs * np.log(probs / REF_PROBS)).sum())


def train_policy(R, beta, steps=300, lr=0.5, batch_size=64, score_fn=true_score):
    """REINFORCE on reward model R with a KL penalty of strength beta."""
    theta = np.zeros((SEQ_LEN, V))     # start equal to the reference policy
    pos = np.arange(SEQ_LEN)
    history = {"rm_reward": [], "true_score": [], "kl": []}

    for _ in range(steps):
        probs = softmax(theta)
        # 1. sample a batch of responses: ys[n, i] is the token id at position i
        ys = np.stack([[rng.choice(V, p=probs[i]) for i in pos] for _ in range(batch_size)])

        # 2. score with the reward model
        rm_reward = R[pos, ys].sum(axis=1)

        # 3. log-ratio of policy to reference for each sample
        log_ratio = np.log(probs[pos, ys]).sum(axis=1) - np.log(REF_PROBS[pos, ys]).sum(axis=1)

        # TODO (Question 3): compute the KL-shaped reward for each sample.
        #   shaped = reward-model reward minus beta times the log-ratio
        raise NotImplementedError("Question 3: compute `shaped`")
        # 4. REINFORCE with a mean baseline
        advantage = shaped - shaped.mean()
        onehots = np.eye(V)[ys]                       # (batch, SEQ_LEN, V)
        grad = (advantage[:, None, None] * (onehots - probs)).mean(axis=0)
        theta += lr * grad

        # bookkeeping: expected true score under the current policy
        probs = softmax(theta)
        tokens = [[VOCAB[j] for j in y] for y in ys]
        history["rm_reward"].append(rm_reward.mean())
        history["true_score"].append(np.mean([score_fn(t) for t in tokens]))
        history["kl"].append(kl_to_ref(probs))

    return softmax(theta), history


def show_policy(probs, title="Policy"):
    print(title)
    print("        " + "  ".join(f"{t:>5}" for t in VOCAB))
    for i in range(SEQ_LEN):
        print(f"pos {i}:  " + "  ".join(f"{probs[i, j]:5.2f}" for j in range(V)))
    p_target = np.prod([probs[i, TOKEN_ID[t]] for i, t in enumerate(TARGET)])
    print(f"P(policy outputs {''.join(TARGET)}) = {p_target:.3f}")

The next cells will fail until you complete **Question 3** below. Come back and run them once `train_policy()` works.

In [ ]:
probs, history = train_policy(R, beta=0.5)
show_policy(probs, "Policy after RLHF (beta = 0.5)")

In [ ]:
def plot_history(history, title):
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
    for ax, key, label in zip(
        axes,
        ["rm_reward", "true_score", "kl"],
        ["Reward-model reward", "True human score (0-3)", "KL to reference"],
    ):
        ax.plot(history[key])
        ax.set_title(label)
        ax.set_xlabel("training step")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


plot_history(history, "beta = 0.5")

---

### Part 4: Reward Hacking

So far the reward model was a faithful copy of the human. In practice it never is. Here the labellers have a bias: they like the token `A` and give each one a 1.5-point bonus, whatever position it's in. The true goal is still `ABC`, but the preference data now carries the bias, and so does the reward model trained on it.

In [ ]:
def biased_labeller_score(seq):
    return true_score(seq) + 1.5 * seq.count("A")


biased_prefs = collect_preferences(N_PAIRS, score_fn=biased_labeller_score)
R_biased = train_reward_model(biased_prefs)
show_table(R_biased, "Reward table learned from biased labellers")

for beta in [0.0, 2.0]:
    probs_b, hist_b = train_policy(R_biased, beta=beta)
    show_policy(probs_b, f"\nPolicy trained on biased reward model, beta = {beta}")
    plot_history(hist_b, f"Biased reward model, beta = {beta}")

---

### Questions

#### 1) Test `human_preference()`
**(POINTS: 1)**

Write calls that check all three cases: `seq1` preferred, `seq2` preferred, and a tie. What fraction of random pairs end in a tie? (Hint: generate 10,000 random pairs and count the ties.)

Answer:


---

#### 2) How much feedback is enough?
**(POINTS: 2)**

Train a reward model with `n_pairs` = 5, 20, 100 and 1000, and report the held-out accuracy for each. Then look at the reward tables. Accuracy levels off quickly, but the size of the table entries keeps growing. Why? Why does that matter for choosing $\beta$?

Answer:


---

#### 3) Implement the KL penalty
**(POINTS: 2)**

In `train_policy()`, replace the `raise` line with the KL-shaped reward from step 3 of Part 3. Then run the Part 3 cells. Does the policy learn to output `ABC`? Remember that it never saw `true_score()`.

Answer:


---

#### 4) Sweep $\beta$
**(POINTS: 2)**

Train on the faithful reward model `R` with β = 0, 0.5, 1, 3 and 10. For each, report the final KL and P(`ABC`). What happens as β → 0, and what happens when β is large? What is the policy at β = ∞?

(Note: with `lr=0.5`, training gets unstable above about β = 10, because the KL term's noisy gradient grows with β. If you try larger values, lower `lr` too.)

Answer:


---

#### 5) Reward hacking
**(POINTS: 2)**

Using the Part 4 results:

- (a) Which response does the β = 0 policy converge to, and what is its true score? Compare the reward-model curve with the true-score curve.
- (b) Sweep β on `R_biased` (try 0, 1, 2, 5 and 10) and record the final true score. Which β does best? Why does the KL penalty help here, even though it never sees the true score?

Answer:


---

#### 6) Toy vs. real
**(POINTS: 1)**

List three ways real RLHF for a language model differs from this lab, and say why each makes the problem harder.

Answer:


---